# Convert Dataset to JSONL Format

The trainer can directly use this dataset format. Use this notebook to convert the dataset from CSV to JSONL format.


In [ ]:
import pandas as pd
import json

dataset_name = "train_dataset_generated_reports.csv"
train_dataset = pd.read_csv(dataset_name)

# Before we had the instructions and expected response in the prompt. Separate them now.
train_dataset['prompt'] = train_dataset['prompt'].apply(lambda x: x.split("### Poročilo")[0].strip())

# Also, add additional instructions to the prompt.
instructions = \
"""Generiraj poročilo o prometu na osnovi vhodnih podatkov, ki se začnejo z '### Vhodni podatki'. Odgovor naj vsebuje samo poročilo. Odgovarjaj v povedih.
Drži se hierarhije dogodkov (od najpomembnejših do najmanj pomembnih): 
- Voznik vozi v napačno smer  
- Zaprta avtocesta 
- Nesreča z zastojem na avtocesti 
- Zastoji zaradi del na avtocesti (ob krajših zastojih se pogosto dogajajo naleti) 
- Zaradi nesreče zaprta glavna ali regionalna cesta 
- Nesreče na avtocestah in drugih cestah 
- Pokvarjena vozila, ko je zaprt vsaj en prometni pas 
- Žival, ki je zašla na vozišče 
- Predmet/razsut tovor na avtocesti 
- Dela na avtocesti, kjer je večja nevarnost naleta (zaprt prometni pas, pred predori, v predorih, …) 
- Zastoj pred Karavankami in mejnimi prehodi 
Pomembno je sporočiti, če voznik ne vozi več v napačno smer ali če je konec zastojev zaradi katere koli prometne nesreče.
"""
train_dataset['prompt'] = instructions + "\n\n" + train_dataset['prompt']

In [ ]:
with open(dataset_name.replace(".csv", ".jsonl"), "w", encoding="utf-8") as f:
    for _, row in train_dataset.iterrows():
        prompt = row['prompt']
        #expected_response = row['generated_response']
        expected_response = row['generated_response']
        json_line = {
            "prompt": prompt,
            "completion": expected_response
        }
        json.dump(json_line, f)